# NCBI Virus H5NX Pipeline

Processes `sequences.fasta` from NCBI Virus to identify complete H5NX genomes and run GenoFLU genotyping. Downloaded all results with Genotype `H5N*` from NCBI Virus on 2/20/26.

**Pipeline stages:**

1. Parse sequences and identify segments (PB2, PB1, PA, HA, NP, NA, MP, NS)
2. Check for Uni12/Uni13 universal primers
3. Find strains with all 8 segments (complete genomes)
4. Deduplicate: keep longest sequence per segment per strain
5. Run GenoFLU for genotype assignment
6. Visualize genotype distribution

**Quality control outputs:**

- `ambiguous_segments_qc.txt` - sequences matching multiple segments
- `missing_strain_qc.txt` - sequences with unparseable strain names
- `duplicate_segments_qc.txt` - strains with multiple sequences per segment

## Imports

In [1]:
import os
import re
import csv
import subprocess
from collections import defaultdict, Counter
from pathlib import Path
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import altair as alt
import pandas as pd

## Setup

We look for specific segment keywords to determine what segment a given sequence represents.

In [2]:
# Primer sequences for quality filtering
UNI12_VARIANTS = ["agcaaaagcagg", "agcgaaagcagg"]
UNI13_RC = "ccttgtttctact"

# Segment identification keywords
# NOTE: MP intentionally avoids bare "M1"/"M2" to prevent false positives (e.g. sample IDs like LBM1021)
SEGMENT_KEYWORDS = {
    'PB2': ['polymerase PB2', 'segment 1', 'PB2 gene', 'PB2)'],
    'PB1': ['polymerase PB1', 'segment 2', 'PB1 gene', 'PB1)', 'PB1-F2'],
    'PA': ['polymerase PA', 'segment 3', 'PA gene', 'PA)', 'PA-X'],
    'HA': ['hemagglutinin', 'segment 4', 'HA gene', 'HA)'],
    'NP': ['nucleocapsid protein', 'nucleoprotein', 'segment 5', 'NP gene', 'NP)'],
    'NA': ['neuraminidase', 'segment 6', 'NA gene', 'NA)'],
    'MP': ['matrix protein', 'segment 7', 'M gene', 'M1 gene', 'M2 gene', 'matrix protein 1', 'matrix protein 2'],
    'NS': ['nonstructural protein', 'segment 8', 'NS1', 'NS2', 'NEP', 'NS gene', 'nuclear export protein']
}

FASTA_PATH = Path("sequences.fasta")
COMPLETE_FASTA = Path("complete_sequences.fasta")
COMPLETE_STRAINS_FILE = Path("complete_strains.txt")
GENOFLU_INPUT = Path("genoflu_input")
GENOFLU_MAP = GENOFLU_INPUT / "strain_id_map.tsv"
GENOFLU_RESULTS = GENOFLU_INPUT / "results" / "results.tsv"
AMBIGUOUS_QC = Path("ambiguous_segments_qc.txt")
MISSING_STRAIN_QC = Path("missing_strain_qc.txt")
DUPLICATE_SEGMENTS_QC = Path("duplicate_segments_qc.txt")

**Constants and paths.** Uni12/Uni13 are universal influenza primers - their presence indicates full-length amplification. Segment keywords map NCBI description text to segment names.

In [3]:
# Helpers

def identify_segment(description):
    """Return (segment, votes, is_ambiguous, is_critical).

    Ambiguous means this description matched keywords from more than one segment.
    Critical means there's a tie at max votes - cannot resolve automatically.
    """
    desc_lower = description.lower()
    votes = {}
    for segment, keywords in SEGMENT_KEYWORDS.items():
        segment_votes = sum(1 for keyword in keywords if keyword.lower() in desc_lower)
        if segment_votes > 0:
            votes[segment] = segment_votes

    if not votes:
        return None, votes, False, False

    max_votes = max(votes.values())
    winners = [seg for seg, v in votes.items() if v == max_votes]
    is_ambiguous = len(votes) > 1
    is_critical = len(winners) > 1
    segment = winners[0] if len(winners) == 1 else None
    return segment, votes, is_ambiguous, is_critical


def extract_strain_name(description):
    """Extract A/host/location/year from NCBI header."""
    # Try 4-digit year first, then 2-digit
    match = re.search(r"A/.+?/\d{4}", description)
    if not match:
        match = re.search(r"A/.+?/\d{2}(?!\d)", description)
    if not match:
        return None
    return match.group(0)


def safe_strain_id(strain):
    """Sanitize strain for GenoFLU headers (no spaces)."""
    safe = re.sub(r"[^A-Za-z0-9/_-]+", "_", strain)
    safe = re.sub(r"_+", "_", safe).strip("_")
    return safe

**Helper functions.** `identify_segment` uses keyword voting to handle ambiguous descriptions - "critical" ambiguity means a tie that can't be resolved. `extract_strain_name` parses standard influenza nomenclature (A/host/location/year).

## Pass 1: Sequence Analysis

Single pass through all sequences to:
- Check primer presence (Uni12 at 5', Uni13 at 3')
- Extract strain name and segment from headers
- Track which segments exist for each strain
- Store sequences for later deduplication (longest wins)

In [4]:
# Pass 1: analyze sequences for primers, segments, and strain completeness
if not FASTA_PATH.exists():
    raise FileNotFoundError(f"Missing input FASTA: {FASTA_PATH}")

strain_segments = defaultdict(set)
strain_total_count = defaultdict(int)
strain_primer_count = defaultdict(int)
# Track all sequences per strain/segment for deduplication (keeps longest)
strain_segment_seqs = defaultdict(lambda: defaultdict(list))  # strain -> segment -> [(record, len, has_primers)]

stats = Counter()
ambiguous_records = []
missing_strain_headers = []

for record in SeqIO.parse(FASTA_PATH, "fasta"):
    stats['total_sequences'] += 1
    header = record.description
    seq = str(record.seq)

    seq_lc = seq.lower()
    has_uni12 = any(seq_lc.startswith(v) for v in UNI12_VARIANTS)
    has_uni13 = seq_lc.endswith(UNI13_RC)
    has_both = has_uni12 and has_uni13
    if has_both:
        stats['with_both_primers'] += 1

    strain = extract_strain_name(header)
    if not strain:
        stats['missing_strain'] += 1
        missing_strain_headers.append(header)
        continue

    segment, votes, ambiguous, critical = identify_segment(header)
    if ambiguous:
        stats['ambiguous_segment'] += 1
        accession = header.split()[0]
        ambiguous_records.append({
            'accession': accession,
            'votes': votes,
            'description': header,
            'critical': critical
        })
    if critical:
        stats['critical_ambiguous'] += 1
    if not segment:
        stats['missing_segment'] += 1
        continue

    strain_segments[strain].add(segment)
    strain_total_count[strain] += 1
    if has_both:
        strain_primer_count[strain] += 1
    
    # Track for deduplication
    strain_segment_seqs[strain][segment].append((record, len(seq), has_both))

    if stats['total_sequences'] % 25000 == 0:
        print(f"  Processed {stats['total_sequences']:,} sequences...")

# Write ambiguous QC file
if ambiguous_records:
    with open(AMBIGUOUS_QC, "w") as out:
        out.write("Sequences with ambiguous segment identification for QC\n")
        out.write("=" * 60 + "\n\n")
        for rec in ambiguous_records:
            marker = "*** CRITICAL ***" if rec['critical'] else ""
            out.write(f"Accession: {rec['accession']} {marker}\n")
            out.write(f"Votes: {rec['votes']}\n")
            out.write(f"Description: {rec['description']}\n\n")

# Write missing strain QC file
if missing_strain_headers:
    with open(MISSING_STRAIN_QC, "w") as out:
        out.write("Sequences where strain name could not be extracted\n")
        out.write("=" * 60 + "\n\n")
        out.write(f"Total: {len(missing_strain_headers):,}\n\n")
        for hdr in missing_strain_headers:
            out.write(f"{hdr}\n")

# Identify and report strains with duplicate segments
duplicate_segment_strains = {}
for strain, segments in strain_segment_seqs.items():
    dups = {seg: len(seqs) for seg, seqs in segments.items() if len(seqs) > 1}
    if dups:
        duplicate_segment_strains[strain] = dups

if duplicate_segment_strains:
    with open(DUPLICATE_SEGMENTS_QC, "w") as out:
        out.write("Strains with multiple sequences per segment (longest will be kept)\n")
        out.write("=" * 60 + "\n\n")
        out.write(f"Total strains with duplicates: {len(duplicate_segment_strains):,}\n\n")
        for strain in sorted(duplicate_segment_strains):
            dups = duplicate_segment_strains[strain]
            out.write(f"{strain}\n")
            for seg, count in sorted(dups.items()):
                out.write(f"  {seg}: {count} sequences\n")
            out.write("\n")

print("\nPASS 1 SUMMARY")
print("---------------")
print(f"Total sequences:              {stats['total_sequences']:,}")
print(f"With both primers:            {stats['with_both_primers']:,} ({100*stats['with_both_primers']/stats['total_sequences']:.1f}%)")
print(f"Missing strain:               {stats['missing_strain']:,}")
print(f"Missing/unknown segment:      {stats['missing_segment']:,}")
print(f"Ambiguous segment:            {stats['ambiguous_segment']:,}")
print(f"  Critical (tied):            {stats['critical_ambiguous']:,}")
print(f"Unique strains found:         {len(strain_segments):,}")
print(f"Strains with duplicate segs:  {len(duplicate_segment_strains):,}")

if ambiguous_records:
    print(f"Ambiguous QC written:         {AMBIGUOUS_QC}")
if missing_strain_headers:
    print(f"Missing strain QC written:    {MISSING_STRAIN_QC}")
if duplicate_segment_strains:
    print(f"Duplicate segments QC written: {DUPLICATE_SEGMENTS_QC}")

  Processed 25,000 sequences...
  Processed 50,000 sequences...
  Processed 75,000 sequences...
  Processed 100,000 sequences...
  Processed 125,000 sequences...
  Processed 150,000 sequences...
  Processed 175,000 sequences...

PASS 1 SUMMARY
---------------
Total sequences:              179,560
With both primers:            6,938 (3.9%)
Missing strain:               69
Missing/unknown segment:      191
Ambiguous segment:            84
  Critical (tied):            3
Unique strains found:         25,588
Strains with duplicate segs:  1,110
Ambiguous QC written:         ambiguous_segments_qc.txt
Missing strain QC written:    missing_strain_qc.txt
Duplicate segments QC written: duplicate_segments_qc.txt


In [5]:
# Compute completeness and gold standard strains
REQUIRED_SEGMENTS = {'PB2', 'PB1', 'PA', 'HA', 'NP', 'NA', 'MP', 'NS'}

segment_counts = Counter(len(segs) for segs in strain_segments.values())
complete_strains = {s for s, segs in strain_segments.items() if segs == REQUIRED_SEGMENTS}

sequences_from_complete_strains = sum(strain_total_count[s] for s in complete_strains)
sequences_from_complete_strains_with_primers = sum(strain_primer_count[s] for s in complete_strains)

# Gold standard: complete genome + all sequences in strain have both primers (for reference)
gold_standard_strains = {
    s for s in complete_strains
    if strain_primer_count[s] == strain_total_count[s]
}

gold_standard_sequences = sum(strain_total_count[s] for s in gold_standard_strains)

print("\nCOMPLETENESS SUMMARY")
print("--------------------")
print(f"Strains with all 8 segments:  {len(complete_strains):,} / {len(strain_segments):,} ({100*len(complete_strains)/len(strain_segments):.1f}%)")
print(f"Sequences from complete strains: {sequences_from_complete_strains:,}")
print(f"  With both primers:           {sequences_from_complete_strains_with_primers:,} ({100*sequences_from_complete_strains_with_primers/max(sequences_from_complete_strains,1):.1f}%)")
print(f"Gold standard strains:         {len(gold_standard_strains):,} (complete + all primers)")
print(f"Gold standard sequences:       {gold_standard_sequences:,} ({100*gold_standard_sequences/stats['total_sequences']:.1f}%)")

print("\nStrain segment counts (how many segments per strain):")
for seg_count in sorted(segment_counts):
    print(f"  {seg_count} segments: {segment_counts[seg_count]:,}")


COMPLETENESS SUMMARY
--------------------
Strains with all 8 segments:  19,958 / 25,588 (78.0%)
Sequences from complete strains: 168,619
  With both primers:           5,969 (3.5%)
Gold standard strains:         236 (complete + all primers)
Gold standard sequences:       2,005 (1.1%)

Strain segment counts (how many segments per strain):
  1 segments: 3,654
  2 segments: 1,042
  3 segments: 302
  4 segments: 119
  5 segments: 95
  6 segments: 134
  7 segments: 284
  8 segments: 19,958


**Completeness check.** A strain is "complete" if it has all 8 segments. "Gold standard" adds the requirement that all sequences have both primers (stricter QC, used for reference only).

In [6]:
# Pass 2: write complete strain sequences (longest per segment) and strain list
print(f"\nWriting {COMPLETE_FASTA} and {COMPLETE_STRAINS_FILE}...")

complete_records = []

# For each complete strain, pick longest sequence per segment
for strain in complete_strains:
    for segment in REQUIRED_SEGMENTS:
        seqs = strain_segment_seqs[strain][segment]
        if seqs:
            # Sort by length descending, pick longest
            best_record, best_len, best_primers = max(seqs, key=lambda x: x[1])
            complete_records.append(best_record)

with open(COMPLETE_FASTA, "w") as fasta_out:
    SeqIO.write(complete_records, fasta_out, "fasta")

extracted_count = len(complete_records)

with open(COMPLETE_STRAINS_FILE, "w") as strains_out:
    strains_out.write("Complete Strains (All 8 segments present, longest sequence per segment)\n")
    strains_out.write("=" * 60 + "\n\n")
    strains_out.write(f"Total strains: {len(complete_strains):,}\n")
    strains_out.write(f"Total sequences: {extracted_count:,}\n")
    avg = extracted_count / max(len(complete_strains), 1)
    strains_out.write(f"Average sequences per strain: {avg:.1f}\n\n")

    strains_out.write("Strain List:\n")
    strains_out.write("-" * 40 + "\n")

    for i, strain in enumerate(sorted(complete_strains), 1):
        seq_count = strain_total_count[strain]
        primer_count = strain_primer_count[strain]
        segments = sorted(strain_segments[strain])
        gold = " [GOLD]" if strain in gold_standard_strains else ""
        dup = " [DUP]" if strain in duplicate_segment_strains else ""
        strains_out.write(f"{i:5d}. {strain}{gold}{dup}\n")
        strains_out.write(f"       Sequences: {seq_count} | With primers: {primer_count} | Segments: {segments}\n\n")

print(f"Extracted {extracted_count:,} sequences from {len(complete_strains):,} complete strains")
print(f"(8 segments per strain, longest sequence selected for duplicates)")


Writing complete_sequences.fasta and complete_strains.txt...
Extracted 159,664 sequences from 19,958 complete strains
(8 segments per strain, longest sequence selected for duplicates)


## GenoFLU Input Preparation

Create per-segment FASTA files matching GenoFLU's expected format: `complete_PB2.fasta`, `complete_PB1.fasta`, etc.

In [7]:
# Prepare GenoFLU input (separate FASTA per segment)
GENOFLU_INPUT.mkdir(exist_ok=True)

# Collect sequences by segment
segment_records = {seg: [] for seg in REQUIRED_SEGMENTS}

for strain in complete_strains:
    for segment in REQUIRED_SEGMENTS:
        seqs = strain_segment_seqs[strain][segment]
        if seqs:
            best_record, best_len, best_primers = max(seqs, key=lambda x: x[1])
            # Create new record with safe strain ID
            safe_id = safe_strain_id(strain)
            new_record = SeqRecord(best_record.seq, id=safe_id, description="")
            segment_records[segment].append(new_record)

# Write per-segment FASTA files
for segment in REQUIRED_SEGMENTS:
    seg_fasta = GENOFLU_INPUT / f"complete_{segment}.fasta"
    with open(seg_fasta, "w") as out:
        SeqIO.write(segment_records[segment], out, "fasta")

# Write strain ID mapping
with open(GENOFLU_MAP, "w") as out_map:
    out_map.write("safe_id\tstrain\n")
    for strain in complete_strains:
        safe_id = safe_strain_id(strain)
        out_map.write(f"{safe_id}\t{strain}\n")

total_seqs = sum(len(recs) for recs in segment_records.values())
print("\nGenoFLU input prepared (per-segment format)")
print(f"Strains: {len(complete_strains):,}")
print(f"Total sequences: {total_seqs:,} ({total_seqs // len(complete_strains)} per strain)")
print(f"Output directory: {GENOFLU_INPUT}/")
for segment in sorted(REQUIRED_SEGMENTS):
    print(f"  complete_{segment}.fasta: {len(segment_records[segment]):,} sequences")


GenoFLU input prepared (per-segment format)
Strains: 19,958
Total sequences: 159,664 (8 per strain)
Output directory: genoflu_input/
  complete_HA.fasta: 19,958 sequences
  complete_MP.fasta: 19,958 sequences
  complete_NA.fasta: 19,958 sequences
  complete_NP.fasta: 19,958 sequences
  complete_NS.fasta: 19,958 sequences
  complete_PA.fasta: 19,958 sequences
  complete_PB1.fasta: 19,958 sequences
  complete_PB2.fasta: 19,958 sequences


## GenoFLU Genotyping

Run GenoFLU-multi on complete genomes. Set `RUN_GENOFLU = True` to execute. Make sure you run `git clone https://github.com/moncla-lab/genoflu-multi` so that the tool is available.

In [8]:
# Optional: run GenoFLU-multi
# Set to True to run. This will take time.
RUN_GENOFLU = False

if RUN_GENOFLU:
    genoflu_dir = Path("/Users/sdshank/Documents/deep-sequencing/choose-reference/genoflu-multi")
    cmd = [
        "/Users/sdshank/software/miniconda3/envs/bioinformatics/bin/python",
        "bin/genoflu-multi.py",
        "-f", "../genoflu_input/",
        "-m",
    ]
    print("Running GenoFLU...")
    subprocess.run(cmd, cwd=str(genoflu_dir), check=True)
    print("GenoFLU complete")
else:
    print("Skipping GenoFLU run. Set RUN_GENOFLU=True to execute.")

Skipping GenoFLU run. Set RUN_GENOFLU=True to execute.


In [9]:
# Analyze GenoFLU results (if present)
if not GENOFLU_RESULTS.exists():
    print(f"Results not found: {GENOFLU_RESULTS}")
else:
    # Load strain mapping
    map_safe_to_strain = {}
    if GENOFLU_MAP.exists():
        with open(GENOFLU_MAP) as f:
            reader = csv.DictReader(f, delimiter='\t')
            for row in reader:
                map_safe_to_strain[row['safe_id']] = row['strain']

    with open(GENOFLU_RESULTS, newline='') as f:
        reader = csv.DictReader(f, delimiter='\t')
        rows = list(reader)

    # Enrich with primer status
    for row in rows:
        safe_id = row['Strain']
        strain = map_safe_to_strain.get(safe_id, safe_id)
        row['original_strain'] = strain
        row['has_primers'] = strain in gold_standard_strains

    genotype_counts = Counter(r['Genotype'] for r in rows)
    known = sum(c for g, c in genotype_counts.items() if 'Not assigned' not in g)
    novel = sum(c for g, c in genotype_counts.items() if 'Not assigned' in g)
    
    # Count by genotype and primer status
    genotype_primer_counts = defaultdict(lambda: {'with_primers': 0, 'without_primers': 0})
    for row in rows:
        g = row['Genotype']
        if row['has_primers']:
            genotype_primer_counts[g]['with_primers'] += 1
        else:
            genotype_primer_counts[g]['without_primers'] += 1

    print("\nGENOFLU SUMMARY")
    print("--------------")
    print(f"Strains analyzed:      {len(rows):,}")
    print(f"Known genotypes:       {known:,} ({100*known/max(len(rows),1):.1f}%)")
    print(f"Novel / unassigned:    {novel:,} ({100*novel/max(len(rows),1):.1f}%)")

    print("\nTop genotypes (total | with primers | without primers):")
    for g, c in genotype_counts.most_common(15):
        wp = genotype_primer_counts[g]['with_primers']
        wop = genotype_primer_counts[g]['without_primers']
        print(f"  {c:5d} | {wp:5d} | {wop:5d}  {g}")

    print("\nGenotype chart skipped (table-only mode).")



GENOFLU SUMMARY
--------------
Strains analyzed:      19,958
Known genotypes:       16,325 (81.8%)
Novel / unassigned:    3,633 (18.2%)

Top genotypes (total | with primers | without primers):
   4616 |     3 |  4613  B3.13
   3555 |     0 |  3555  D1.1
   2879 |   142 |  2737  Not assigned: Only 0 segments >98.0% match found of total 8 segments in input file
   1812 |    80 |  1732  B3.2
    868 |     0 |   868  B2.1
    704 |     0 |   704  B4.1
    659 |     0 |   659  B1.1
    580 |     0 |   580  B1.2
    537 |     0 |   537  A1
    498 |     0 |   498  B1.3
    464 |     0 |   464  B3.6
    387 |     0 |   387  A2
    346 |     0 |   346  A3
    249 |     0 |   249  D1.3
    218 |     0 |   218  B2.2

Genotype chart skipped (table-only mode).


In [10]:
df = pd.DataFrame(rows)

# Drop unassigned calls
df = df[~df["Genotype"].str.startswith("Not assigned:", na=False)].copy()

genotype_table = (
    df.groupby(["Genotype", "has_primers"])
      .size()
      .unstack(fill_value=0)
      .rename(columns={True: "with_primers", False: "without_primers"})
)

# Ensure both columns exist
if "with_primers" not in genotype_table.columns:
    genotype_table["with_primers"] = 0
if "without_primers" not in genotype_table.columns:
    genotype_table["without_primers"] = 0

genotype_table["total"] = genotype_table["with_primers"] + genotype_table["without_primers"]
genotype_table["pct_with_primers"] = (
    100 * genotype_table["with_primers"] / genotype_table["total"]
).round(2)
genotype_table["pct_without_primers"] = (
    100 * genotype_table["without_primers"] / genotype_table["total"]
).round(2)

# Final columns only
genotype_table = (
    genotype_table[["total", "pct_with_primers", "pct_without_primers"]]
    .sort_values("total", ascending=False)
    .reset_index()
)

display(genotype_table.head(20))
genotype_table.to_csv("genotype_table_filtered.tsv", sep="\t", index=False)
print("Saved: genotype_table_filtered.tsv")


has_primers,Genotype,total,pct_with_primers,pct_without_primers
0,B3.13,4616,0.06,99.94
1,D1.1,3555,0.00,100.00
2,B3.2,1812,4.42,95.58
3,B2.1,868,0.00,100.00
4,B4.1,704,0.00,100.00
5,B1.1,659,0.00,100.00
6,B1.2,580,0.00,100.00
7,A1,537,0.00,100.00
8,B1.3,498,0.00,100.00
9,B3.6,464,0.00,100.00


Saved: genotype_table_filtered.tsv


## GenoFLU Lineages by Segment

Parse the GenoFLU per-segment lineage calls (`SEGMENT:lineage`) and summarize lineage composition for each segment, split by primer status.


In [11]:
# Segment lineage analysis from GenoFLU results
lineage_col = 'Genotype List Used, >=98.0%'
segment_order = ['PB2', 'PB1', 'PA', 'HA', 'NP', 'NA', 'MP', 'NS']

if 'rows' not in globals() or not rows:
    print('No GenoFLU rows found in memory. Run the GenoFLU results analysis cell first.')
else:
    parsed = []
    parse_errors = 0

    for row in rows:
        assignment_text = (row.get(lineage_col) or '').strip()
        if not assignment_text:
            continue

        for token in [t.strip() for t in assignment_text.split(',') if t.strip()]:
            if ":" not in token:
                parse_errors += 1
                continue

            segment, lineage = token.split(':', 1)
            segment = segment.strip().upper()
            lineage = lineage.strip()

            if segment not in segment_order or not lineage:
                parse_errors += 1
                continue

            parsed.append({
                'Strain': row.get('original_strain', row.get('Strain')),
                'segment': segment,
                'lineage': lineage,
                'has_primers': bool(row.get('has_primers', False)),
            })

    lineage_df = pd.DataFrame(parsed)

    if lineage_df.empty:
        print('No segment lineage assignments were parsed from GenoFLU results.')
    else:
        lineage_summary = (
            lineage_df.groupby(['segment', 'lineage', 'has_primers'])
            .size()
            .unstack(fill_value=0)
            .rename(columns={True: 'with_primers', False: 'without_primers'})
            .reset_index()
        )

        if 'with_primers' not in lineage_summary.columns:
            lineage_summary['with_primers'] = 0
        if 'without_primers' not in lineage_summary.columns:
            lineage_summary['without_primers'] = 0

        lineage_summary['total'] = lineage_summary['with_primers'] + lineage_summary['without_primers']
        lineage_summary['pct_with_primers'] = (
            100 * lineage_summary['with_primers'] / lineage_summary['total']
        ).round(2)
        lineage_summary['pct_without_primers'] = (
            100 * lineage_summary['without_primers'] / lineage_summary['total']
        ).round(2)

        lineage_summary['segment'] = pd.Categorical(
            lineage_summary['segment'], categories=segment_order, ordered=True
        )
        lineage_summary = lineage_summary.sort_values(
            ['segment', 'total'], ascending=[True, False]
        ).reset_index(drop=True)

        print('GENOFLU LINEAGE SUMMARY BY SEGMENT')
        print('----------------------------------')
        print(f'Rows in results.tsv:           {len(rows):,}')
        print(f'Parsed segment assignments:    {len(lineage_df):,}')
        print(f'Unique segment lineages:       {lineage_summary[["segment", "lineage"]].drop_duplicates().shape[0]:,}')
        print(f'Unparsed tokens skipped:       {parse_errors:,}')

        lineage_summary.to_csv('lineage_table_by_segment.tsv', sep='\t', index=False)
        print('Saved: lineage_table_by_segment.tsv')
lineage_summary

GENOFLU LINEAGE SUMMARY BY SEGMENT
----------------------------------
Rows in results.tsv:           19,958
Parsed segment assignments:    134,236
Unique segment lineages:       111
Unparsed tokens skipped:       0
Saved: lineage_table_by_segment.tsv


has_primers,segment,lineage,without_primers,with_primers,total,pct_with_primers,pct_without_primers
0,PB2,am2.2,5419,3,5422,0.06,99.94
1,PB2,am24,3879,0,3879,0.00,100.00
2,PB2,am2.1,1954,81,2035,3.98,96.02
3,PB2,am1.1,1253,0,1253,0.00,100.00
4,PB2,am1.2,1106,0,1106,0.00,100.00
...,...,...,...,...,...,...,...
106,NS,am1.3,5,0,5,0.00,100.00
107,NS,am5,5,0,5,0.00,100.00
108,NS,am1.4,4,0,4,0.00,100.00
109,NS,am4,3,0,3,0.00,100.00
